In [4]:
!pip install -U langchain-chroma

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import json

with open("C:\\Users\\Nitu28\\startup-policy-copilot\\eval\\eval_set.json", encoding="utf-8") as f:
    eval_qs = json.load(f)

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma(
    persist_directory="C:\\Users\\Nitu28\\startup-policy-copilot\\data\\index",
    embedding_function=embedding_model
)

retriever = db.as_retriever(search_kwargs={"k": 3})

def search_policy_docs(query: str) -> str:
    docs = retriever.get_relevant_documents(query)
    answer = ""
    for doc in docs:
        answer += doc.page_content[:800] + "\n"
        answer += f"source: {doc.metadata['source']}\n\n"
    return answer

from difflib import SequenceMatcher

scores = []

for item in eval_qs:
    question = item["question"]
    keywords = item["expected"]
    answer = search_policy_docs(question).lower()
    
    matched = any(kw.lower() in answer for kw in keywords)
    has_citation = "source:" in answer
    score = 1 if matched and has_citation else 0
    scores.append(score)
    
    print(f"Q: {question}")
    print(f"Keywords: {keywords}")
    print(f"Got: {answer[:500]}")
    print(f"Keywords matched: {matched}")
    print(f"Citation found: {has_citation}")
    print(f"Match: {score}\n")

accuracy = sum(scores) / len(scores)
print(f"Overall Evaluation Accuracy: {accuracy*100:.2f}%")


Q: What is the definition of a startup under DPIIT?
Keywords: ['dpiit', 'startup', 'gsr']
Got: innovation, development or improvement of products or processes or services, or its scalability in terms of employment generation or we alth creation. (iii) the dpiit may, after calling for such documen ts or information and making such enquires, as it may deem fit, — (a) recognise the eligible entity as startup; or (b) reject the application by providing reasons. certification for the purposes of section 80-iac of the act 3. a startup being a private limited company or lim ited liability
source
Keywords matched: True
Citation found: True
Match: 1

Q: What are tax benefits under section 56?
Keywords: ['angel tax', 'section 56']
Got: declaration by a startup for exemption under section 56(2)(viib) of the income tax act, 1961 <to be issued on company letter head > i, _____________________________ son/ daughter of ______________________ having permanent account number (pan) _____________________